# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library, referencing dataset entities by their `@id`.

### Dataset Source
The dataset is described by a [Croissant schema (JSON-LD)](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields, referencing their `@id` values.

**Note:** The dataset may contain multiple record sets; below, we list their `@id`s and fields.

In [ ]:
# List all record set @id's in the dataset
rs = dataset.record_sets
print(f"Number of record sets: {len(rs)}\n")
for record_set in rs:
    print(f"RecordSet @id: {record_set['@id']}")
    field_ids = [field['@id'] for field in record_set.get('field', [])]
    print(f"  Fields: {field_ids if field_ids else 'No fields listed in schema'}\n")

### Preview Records from Each Record Set
For each record set, we print a couple of example records (referenced by the record set's `@id`).

In [ ]:
# Explore records for each record set by @id
for rs_meta in dataset.record_sets:
    rs_id = rs_meta['@id']
    print(f"\n=== Records from RecordSet @id = {rs_id} ===")
    # Print first two records (if available)
    it = dataset.records(record_set=rs_id)
    try:
        print(next(it))
        print(next(it))
    except StopIteration:
        print("No records found for this record set.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use only `@id` values to reference entities.

Below, select record set(s) by `@id` and extract to DataFrame.

In [ ]:
# Gather all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for RecordSet @id: {record_set_id} ({dataframes[record_set_id].shape[0]} rows)")
        print(f"Fields (@id): {list(dataframes[record_set_id].columns)}\n")
    else:
        print(f"No records loaded for RecordSet @id: {record_set_id}\n")
# Select the first available record set for further analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. Reference all fields by their `@id` in code and text.

Choose a numeric field and group field (by their `@id`) from those listed above.

In [ ]:
# Example setup: adjust these @id values to match your dataset structure

# Use these if known: otherwise, you may need to inspect the DataFrame columns to identify valid options
numeric_field_id = None
group_field_id = None
if main_record_set_id and main_record_set_id in dataframes:
    cols = list(dataframes[main_record_set_id].columns)
    # Attempt to auto-select a numeric field by inspecting types
    for c in cols:
        if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][c]):
            numeric_field_id = c
            break
    # Choose another column as group field (if available)
    for c in cols:
        if c != numeric_field_id:
            group_field_id = c
            break

    print(f"Using numeric field (@id): {numeric_field_id}")
    print(f"Using group field (@id): {group_field_id}")

    if numeric_field_id:
        # Filter records based on numeric field
        threshold = dataframes[main_record_set_id][numeric_field_id].mean()
        filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold].copy()
        print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by the group field (if available)
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
else:
    print("No suitable record set and fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, ensuring to reference columns via their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id], kde=True)
    plt.xlabel(f"{numeric_field_id}")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.tight_layout()
    plt.show()

    if group_field_id and group_field_id in dataframes[main_record_set_id].columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=dataframes[main_record_set_id][group_field_id], y=dataframes[main_record_set_id][numeric_field_id])
        plt.xlabel(f"{group_field_id}")
        plt.ylabel(f"{numeric_field_id}")
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to explore, filter, and visualize the FAIR² dataset.

- All entities (record sets, fields) were referenced by their `@id`.
- The approach enables scalable, schema-consistent data extraction and mining.

Further analysis could include more advanced machine learning, missing value imputation, and reporting on specific adoption predictors in the context of rangeland management.